[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap09/cap09.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="19%"}](https://github.com/fzampirolli/pdi-vc)


## 💻 **Parte Prática com Exercícios de Programação**

A presente lista de exercícios de programação (EP) consolida as formulações teóricas apresentadas ao longo do Capítulo 9 por meio de uma trilha prática aplicada. Os exercícios isolam as grandezas intermediárias de um *pipeline* de aprendizado profundo — permitindo validar manualmente cada etapa do raciocínio sem depender de bibliotecas pesadas nem de tempo excessivo de treinamento.

---

### EP09_01 🟢 Profundidade da Rede e Redução Espacial

No Projeto Prático 1 deste capítulo, foi apresentada uma CNN de duas camadas convolucionais. Ao adicionar uma terceira camada convolucional, a redução espacial causada pelas operações de *pooling* consecutivas torna-se crítica, especialmente em mapas de entrada pequenos. Você foi encarregado de implementar uma função que calcula a dimensão espacial e a quantidade de parâmetros ao adicionar uma nova camada convolucional com *padding* e *pooling* configuráveis.

#### 📋 Diretrizes de Implementação

1. **Entrada:** Ler a dimensão da imagem quadrada de entrada $H$ ($H \times H$), o número de canais iniciais $C_{in}$, e o número de filtros $F_1, F_2, F_3$ para três camadas convolucionais sucessivas com *kernels* $3 \times 3$.
2. **Flag de Pooling:** Ler três inteiros binários $p_1, p_2, p_3 \in \{0, 1\}$ indicando se haverá *max-pooling* $2 \times 2$ (stride 2) após cada respectiva camada.
3. **Cálculo de Dimensão:** Para cada camada $i \in \{1, 2, 3\}$, a convolução utiliza *padding* 1 (mantendo dimensão), e o *pooling* reduz a dimensão para $\lfloor H_{atual} / 2 \rfloor$ se $p_i = 1$.
4. **Cálculo de Parâmetros:** 
   $$\text{Params}_i = (3 \times 3 \times C_{in, i} \times F_i) + F_i$$
   (com viés).
5. **Saída:** Imprimir a dimensão espacial final $H_{final}$, o total de parâmetros das 3 camadas convolucionais e o status `VALIDO` se $H_{final} \ge 1$ ou `INVALIDO` se $H_{final} < 1$.

#### 📌 Restrições Computacionais

* **Kernels Fixos $3 \times 3$:** Todas as camadas convolucionais usam filtros de dimensão $3 \times 3$ com *padding* igual a 1.
* **Comportamento do Pooling:** Janela $2 \times 2$ com *stride* 2.
* **Canais de Entrada:** A primeira camada recebe $C_{in}$ canais; a segunda recebe $F_1$; a terceira recebe $F_2$.

#### 🧠 Fundamentação Teórica

| Elemento | Papel na arquitetura profunda |
|---|---|
| Profundidade | Permite aprender abstrações mais complexas em detrimento de maior custo computacional |
| *Padding* | Evita a degradação prematura da resolução espacial em imagens pequenas |
| Colapso Espacial | *Poolings* excessivos em entradas pequenas (ex.: $8 \times 8$) podem reduzir a dimensão a zero |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiros $H$ e $C_{in}$.
* Linha 2: Inteiros $F_1$, $F_2$ e $F_3$.
* Linha 3: Inteiros $p_1$, $p_2$ e $p_3$ (flags $0$ ou $1$).

**Saída:**
* Linha 1: `Dimensao final: H_final x H_final`
* Linha 2: `Parametros Conv: T`
* Linha 3: `Status: VALIDO` ou `INVALIDO` 

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 8 1<br>8 16 32<br>1 1 0 | Dimensao final: 2 x 2<br>Parametros Conv: 5864<br>Status: VALIDO | Três camadas em entrada $8 \times 8$ com 2 poolings mantêm dimensão $2 \times 2$. |
| 8 1<br>8 16 32<br>1 1 1 | Dimensao final: 1 x 1<br>Parametros Conv: 5864<br>Status: VALIDO | Três poolings reduzem $8 \times 8 \to 4 \times 4 \to 2 \times 2 \to 1 \times 1$. |

In [1]:
#| label: fig-09-sim-ep01
#| fig-cap: "Simulador: Profundidade e Dimensão Espacial"
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-ep0901" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Profundidade e Colapso Espacial</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 impacto do pooling</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Alterne o uso de pooling em cada camada e veja a redução espacial de uma entrada 8×8.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;display:flex;gap:12px;justify-content:center;flex-wrap:wrap;">
      <label><input type="checkbox" id="ep0901_p1" checked> Pool 1</label>
      <label><input type="checkbox" id="ep0901_p2" checked> Pool 2</label>
      <label><input type="checkbox" id="ep0901_p3"> Pool 3</label>
    </div>
    <div id="ep0901_debug" style="background:#e3f2fd;border-radius:8px;padding:12px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var p1 = root.querySelector('#ep0901_p1'), p2 = root.querySelector('#ep0901_p2'), p3 = root.querySelector('#ep0901_p3');
    var dbg = root.querySelector('#ep0901_debug');
    function render(){
      var h = 8;
      if(p1.checked) h = Math.floor(h/2);
      if(p2.checked) h = Math.floor(h/2);
      if(p3.checked) h = Math.floor(h/2);
      var params = (3*3*1*8+8) + (3*3*8*16+16) + (3*3*16*32+32);
      dbg.innerHTML = 'Entrada: 8×8 → Saída Final: <b>' + h + '×' + h + '</b><br>Total Parâmetros Conv: <b>' + params + '</b> | Status: ' + (h>=1 ? '<b style="color:green">VALIDO</b>' : '<b style="color:red">INVALIDO</b>');
    }
    [p1,p2,p3].forEach(function(e){ e.addEventListener('change', render); });
    render();
  }
  function tryInit(){ var r = document.getElementById('sim-ep0901'); if(r) init(r); else setTimeout(tryInit, 200); }
  tryInit();
})();
</script>
''')

---

### EP09_02 🟢 Curva de Eficiência da Transferência de Aprendizado

Na transferência de aprendizado, a vantagem de reaproveitar um extrator congelado diminui à medida que o volume de dados do domínio de destino cresce. Este exercício calcula teoricamente a acurácia esperada comparando as duas abordagens com base em uma função de saturação logarítmica de amostragem.

#### 📋 Diretrizes de Implementação

1. **Entrada:** Ler o inteiro $N$ (número de exemplos disponíveis no domínio B, onde $N \in \{5, 10, 20, 40, 80\}$).
2. **Modelo da Transferência:** A acurácia estimada por transferência com extrator congelado é modelada por:
   $$Acc_{trans}(N) = \min\left(0{,}95,\; 0{,}50 + 0{,}15 \cdot \log_2(N)\right)$$
3. **Modelo Treinado do Zero:** A acurácia estimada treinando do zero é modelada por:
   $$Acc_{zero}(N) = \min\left(0{,}95,\; 0{,}10 + 0{,}25 \cdot \log_2(N)\right)$$
4. **Decisão de Paridade:** Calcular a diferença $\Delta = Acc_{trans}(N) - Acc_{zero}(N)$. Se $\Delta \le 0{,}05$, considerar que a abordagem "Do Zero" tornou-se **competitiva**.
5. **Saída:** Imprimir as acurácias com 4 casas decimais e o resultado da comparação.

#### 📌 Restrições Computacionais

* **Limitação Superior:** Acurácias são limitadas ao teto de $0{,}95$ ($95\%$).
* **Precisão de Logaritmo:** Utilize logaritmo na base 2 ($\log_2$).

#### 🧠 Fundamentação Teórica

| Abordagem | Comportamento com Poucos Dados | Comportamento com Muitos Dados |
|---|---|---|
| Transferência | Alta acurácia inicial (aproveita filtros genéricos do domínio A) | Satura rapidamente devido ao congelamento |
| Do Zero | Baixa acurácia (sobreajuste severo devido à falta de dados) | Cresce rapidamente e iguala ou supera a transferência |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $N$.

**Saída:**
* Linha 1: `Transferencia: Acc_trans` (4 casas decimais).
* Linha 2: `Do Zero: Acc_zero` (4 casas decimais).
* Linha 3: `Status: COMPETITIVO` se $\Delta \le 0{,}05$ ou `Status: VANTAGEM_TRANSFERENCIA` caso contrário.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 5 | Transferencia: 0.8483<br>Do Zero: 0.6805<br>Status: VANTAGEM_TRANSFERENCIA | Com 5 amostras, a transferência tem ampla vantagem. |
| 40 | Transferencia: 0.9500<br>Do Zero: 0.9500<br>Status: COMPETITIVO | Com 40 amostras, o treino do zero atinge paridade. |

In [2]:
#| label: fig-09-sim-ep02
#| fig-cap: "Simulador: Transferência vs. Treino do Zero"
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-ep0902" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Limiar de Transferência de Dados</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 amostragem x acurácia</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Arraste o número de amostras N e observe quando a curva "Do Zero" alcança a "Transferência".</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Amostras N</label>
        <span id="ep0902_nv" style="font-family:monospace;font-weight:bold;color:#2980b9;">5</span>
      </div>
      <input id="ep0902_n" style="width:100%;accent-color:#2980b9;" max="80" min="5" step="5" type="range" value="5">
    </div>
    <div id="ep0902_debug" style="background:#e3f2fd;border-radius:8px;padding:12px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var nEl = root.querySelector('#ep0902_n'), nvEl = root.querySelector('#ep0902_nv');
    var dbg = root.querySelector('#ep0902_debug');
    function render(){
      var n = parseInt(nEl.value);
      nvEl.textContent = n;
      var accT = Math.min(0.95, 0.50 + 0.15 * Math.log2(n));
      var accZ = Math.min(0.95, 0.10 + 0.25 * Math.log2(n));
      var comp = (accT - accZ) <= 0.05;
      dbg.innerHTML = 'Acc Transferência: <b>' + accT.toFixed(4) + '</b> | Acc Do Zero: <b>' + accZ.toFixed(4) + '</b><br>Status: ' + (comp ? '<b style="color:green">COMPETITIVO</b>' : '<b style="color:orange">VANTAGEM TRANSFERENCIA</b>');
    }
    nEl.addEventListener('input', render);
    render();
  }
  function tryInit(){ var r = document.getElementById('sim-ep0902'); if(r) init(r); else setTimeout(tryInit, 200); }
  tryInit();
})();
</script>
''')

---

### EP09_03 🟡 Ajuste Fino Parcial (*Fine-Tuning*) de Parâmetros

Ajustar parcialmente uma rede envolve congelar as primeiras camadas convolucionais (que extraem bordas genéricas) e descongelar camadas intermediárias (`conv2`) e a cabeça classificadora. Você deve calcular o total de parâmetros **treináveis** versus **congelados** em diferentes configurações de descongelamento.

#### 📋 Diretrizes de Implementação

1. **Arquitetura Base (CNN de 2 camadas):**
   * `conv1`: $1 \to 8$ canais, kernel $3 \times 3$, com viés ($80$ parâmetros).
   * `conv2`: $8 \to 16$ canais, kernel $3 \times 3$, com viés ($1168$ parâmetros).
   * `fc1`: entrada $16 \times 2 \times 2 = 64 \to 32$ neurônios, com viés ($2080$ parâmetros).
   * `fc2`: $32 \to 5$ classes, com viés ($165$ parâmetros).
2. **Modo de Treinamento:** Ler a string de modo: `CONGELADO` (apenas `fc1` e `fc2` treináveis), `PARCIAL` (`conv2`, `fc1` e `fc2` treináveis) ou `TOTAL` (todas as camadas treináveis).
3. **Cálculo:** Somar os parâmetros treináveis $P_{trein}$ e congelados $P_{cong}$ de acordo com a estratégia.
4. **Saída:** Imprimir os totais e a porcentagem de parâmetros sendo atualizados.

#### 📌 Restrições Computacionais

* **Cálculo exato de pesos e viés:** $P_{\text{conv}} = k_h \cdot k_w \cdot c_{in} \cdot c_{out} + c_{out}$; $P_{\text{fc}} = in \cdot out + out$.

#### 🧠 Fundamentação Teórica

| Modo | Camadas Treináveis | Aplicação Prática |
|---|---|---|
| `CONGELADO` | Apenas cabeça FC | Pouquíssimos dados no domínio alvo; previne sobreajuste severo |
| `PARCIAL` | Camadas profundas + FC | Dados moderados; adapta características de médio nível |
| `TOTAL` | Todas as camadas | Abundância de dados; substitui completamente os filtros pré-treinados |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: String do modo (`CONGELADO`, `PARCIAL` ou `TOTAL`).

**Saída:**
* Linha 1: `Treinaveis: P_trein`
* Linha 2: `Congelados: P_cong`
* Linha 3: `Percentual Treinavel: PCT%` (2 casas decimais)

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| CONGELADO | Treinaveis: 2245<br>Congelados: 1248<br>Percentual Treinavel: 64.26% | Apaziguamento de memória/gradiente travando a base convolucional. |
| PARCIAL | Treinaveis: 3413<br>Congelados: 80<br>Percentual Treinavel: 97.71% | Descongela `conv2` para reajuste de características. |

In [3]:
#| label: fig-09-sim-ep03
#| fig-cap: "Simulador: Fine-Tuning Parcial"
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-ep0903" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Fine-Tuning Parcial</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 gradientes ativos</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Selecione a estratégia de congelamento e veja a proporção de parâmetros treináveis.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;display:flex;gap:12px;justify-content:center;">
      <button id="ep0903_b1" style="padding:6px 12px;cursor:pointer;">CONGELADO</button>
      <button id="ep0903_b2" style="padding:6px 12px;cursor:pointer;">PARCIAL</button>
      <button id="ep0903_b3" style="padding:6px 12px;cursor:pointer;">TOTAL</button>
    </div>
    <div id="ep0903_debug" style="background:#e3f2fd;border-radius:8px;padding:12px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var dbg = root.querySelector('#ep0903_debug');
    var pConv1=80, pConv2=1168, pFc=2245, total=3493;
    function calc(modo){
      var tr=0, cg=0;
      if(modo==='CONGELADO'){ tr=pFc; cg=pConv1+pConv2; }
      else if(modo==='PARCIAL'){ tr=pFc+pConv2; cg=pConv1; }
      else { tr=total; cg=0; }
      var pct = (tr/total)*100;
      dbg.innerHTML = 'Modo: <b>'+modo+'</b><br>Treináveis: <b>'+tr+'</b> | Congelados: <b>'+cg+'</b><br>Percentual Treinável: <b>'+pct.toFixed(2)+'%</b>';
    }
    root.querySelector('#ep0903_b1').addEventListener('click', function(){ calc('CONGELADO'); });
    root.querySelector('#ep0903_b2').addEventListener('click', function(){ calc('PARCIAL'); });
    root.querySelector('#ep0903_b3').addEventListener('click', function(){ calc('TOTAL'); });
    calc('CONGELADO');
  }
  function tryInit(){ var r = document.getElementById('sim-ep0903'); if(r) init(r); else setTimeout(tryInit, 200); }
  tryInit();
})();
</script>
''')

---

### EP09_04 🟡 Disparidade e Triangulação Estereoscópica

Na visão estereoscópica, a profundidade $Z$ de um ponto em relação ao par de câmeras guarda uma relação inversamente proporcional à sua disparidade $d$ (deslocamento em pixels entre as vistas esquerda e direita):
$$Z = \frac{f \cdot B}{d}$$
onde $f$ é a distância focal (em pixels) e $B$ é a linha de base (*baseline*, em cm).

#### 📋 Diretrizes de Implementação

1. **Entrada:** Ler os valores reais de $f$ (pixels) e $B$ (cm).
2. **Disparidades:** Ler o inteiro $M$ (quantidade de pontos) e, em seguida, $M$ valores reais representando as disparidades $d_i$ calculadas para diferentes regiões da imagem.
3. **Cálculo da Profundidade:** Para cada disparidade $d_i$:
   * Se $d_i \le 0{,}1$, atribuir a profundidade como `INF` (indefinida/infinito) devido ao limite prático de triangulação.
   * Caso contrário, calcular $Z_i = (f \cdot B) / d_i$.
4. **Saída:** Para cada ponto, imprimir a disparidade $d_i$ e a profundidade $Z_i$ em cm com 2 casas decimais (ou `INF`).

#### 📌 Restrições Computacionais

* **Prevenção de Divisão por Zero:** Disparidades menores ou iguais a $0{,}1$ pixel devem ser tratadas como limiar de alcance infinito (`INF`).

#### 🧠 Fundamentação Teórica

| Disparidade $d$ | Distância $Z$ | Interpretação Visual |
|---|---|---|
| Alta (ex.: $> 20\text{ px}$) | Pequena (Objeto Próximo) | Deslocamento visível pronunciado entre as imagens |
| Baixa (ex.: $< 5\text{ px}$) | Grande (Objeto Distante) | Praticamente a mesma posição em ambas as câmeras |
| Próxima de Zero | Infinito / Indefinido | Fora do alcance útil de profundidade estéreo |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Reais $f$ e $B$.
* Linha 2: Inteiro $M$.
* Próximas $M$ linhas: Real $d_i$.

**Saída:**
* $M$ linhas no formato: `Disparidade: d_i px -> Profundidade: Z_i cm` (2 casas decimais ou `INF`).

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 700 6.0<br>3<br>22.0<br>4.0<br>0.05 | Disparidade: 22.00 px -> Profundidade: 190.91 cm<br>Disparidade: 4.00 px -> Profundidade: 1050.00 cm<br>Disparidade: 0.05 px -> Profundidade: INF | Objeto a 22px está a ~1,9m; a 4px está a 10,5m. |

In [4]:
#| label: fig-09-sim-ep04
#| fig-cap: "Simulador: Triangulação Estereoscópica Z = f·B/d"
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-ep0904" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Visão Estereoscópica</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 relação inversa Z(d)</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste a disparidade d e veja a profundidade estimada Z ser calculada.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Disparidade d (px)</label>
        <span id="ep0904_dv" style="font-family:monospace;font-weight:bold;color:#2980b9;">12.0</span>
      </div>
      <input id="ep0904_d" style="width:100%;accent-color:#2980b9;" max="40" min="0" step="0.5" type="range" value="12">
    </div>
    <div id="ep0904_debug" style="background:#e3f2fd;border-radius:8px;padding:12px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var dEl = root.querySelector('#ep0904_d'), dvEl = root.querySelector('#ep0904_dv');
    var dbg = root.querySelector('#ep0904_debug');
    var f = 700, B = 6.0;
    function render(){
      var d = parseFloat(dEl.value);
      dvEl.textContent = d.toFixed(1);
      var zTxt = d <= 0.1 ? 'INF' : ((f * B) / d).toFixed(2) + ' cm';
      dbg.innerHTML = 'f = 700px, B = 6cm | d = <b>' + d.toFixed(1) + ' px</b> → Profundidade Z = <b>' + zTxt + '</b>';
    }
    dEl.addEventListener('input', render);
    render();
  }
  function tryInit(){ var r = document.getElementById('sim-ep0904'); if(r) init(r); else setTimeout(tryInit, 200); }
  tryInit();
})();
</script>
''')

---

### EP09_05 🔴 Efeito da Profundidade de Congelamento (*Freeze*) no YOLO

Ao aplicar transferência de aprendizado com modelos da família YOLO, ajusta-se o parâmetro `freeze` para indicar quantas camadas do *backbone* permanecerão intocadas. Quando os domínios de origem e destino são muito dissimilares, congelar camadas demais reduz a adaptabilidade do modelo, enquanto congelar de menos pode causar sobreajuste se o conjunto de dados for pequeno.

#### 📋 Diretrizes de Implementação

1. **Entrada:** Ler o inteiro $F \in \{0, 5, 10, 15\}$ indicando a quantidade de camadas congeladas no *backbone*.
2. **Modelo de Desempenho ($\text{mAP}_{50}$):** Em um cenário de domínio distante (ex.: COCO $\to$ Formas Geométricas Sintéticas com poucos dados), a métrica $\text{mAP}_{50}$ estimada comporta-se conforme a parábola de ajuste:
   $$\text{mAP}_{50}(F) = \max\left(0{,}0,\; 0{,}88 - 0{,}0025 \cdot (F - 5)^2\right)$$
3. **Cálculo da Taxa de Parâmetros Treináveis:** Considerando um *backbone* com 15 camadas de 100 mil parâmetros cada e uma cabeça de detecção com 500 mil parâmetros (total: $2{,}0$ milhões):
   * Parâmetros congelados: $P_{\text{cong}} = F \times 100.000$.
   * Parâmetros treináveis: $P_{\text{trein}} = 2.000.000 - P_{\text{cong}}$.
4. **Saída:** Imprimir o $mAP_{50}$ esperado e o número de parâmetros treináveis em milhões ($M$).

#### 📌 Restrições Computacionais

* **Valores válidos de $F$:** $F \in [0, 15]$.
* **Formatação de Float:** Exibir $\text{mAP}_{50}$ com 3 casas decimais e parâmetros treináveis em milhões com 2 casas decimais.

#### 🧠 Fundamentação Teórica

| Camadas Congeladas ($F$) | Efeito no *Backbone* | Resultado Didático Esperado |
|---|---|---|
| $F = 0$ (Nenhum) | Todas as camadas treinam | Risco leve de destruir pesos úteis com poucos dados |
| $F = 5$ (Ideal) | Congela bordas de baixo nível | Ponto ótimo: preserva extratores básicos e adapta alto nível |
| $F = 15$ (Máximo) | Congela todo o *backbone* | Inflexível: não adapta representações para o novo domínio |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $F$.

**Saída:**
* Linha 1: `mAP50 Estimado: MAP` (3 casas decimais)
* Linha 2: `Parametros Treinaveis: P M` (em milhões, 2 casas decimais)

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 5 | mAP50 Estimado: 0.880<br>Parametros Treinaveis: 1.50 M | Ponto ideal de congelamento parcial. |
| 15 | mAP50 Estimado: 0.630<br>Parametros Treinaveis: 0.50 M | Congelamento excessivo prejudica o ajuste em domínios distantes. |

In [5]:
#| label: fig-09-sim-ep05
#| fig-cap: "Simulador: Freeze Backbone no YOLO"
#| echo: false
#| output: true

from IPython.display import HTML
HTML('''
<div id="sim-ep0905" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Parameter Freezing no YOLO</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔴 mAP50 vs. freeze</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste o número de camadas congeladas no backbone e observe o mAP50 e o volume de parâmetros.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Camadas Congeladas (F)</label>
        <span id="ep0905_fv" style="font-family:monospace;font-weight:bold;color:#2980b9;">5</span>
      </div>
      <input id="ep0905_f" style="width:100%;accent-color:#2980b9;" max="15" min="0" step="1" type="range" value="5">
    </div>
    <div id="ep0905_debug" style="background:#e3f2fd;border-radius:8px;padding:12px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var fEl = root.querySelector('#ep0905_f'), fvEl = root.querySelector('#ep0905_fv');
    var dbg = root.querySelector('#ep0905_debug');
    function render(){
      var F = parseInt(fEl.value);
      fvEl.textContent = F;
      var map = Math.max(0.0, 0.88 - 0.0025 * Math.pow(F - 5, 2));
      var pTr = (2000000 - F * 100000) / 1000000;
      dbg.innerHTML = 'Freeze = <b>' + F + '</b> camadas<br>mAP50 Estimado: <b>' + map.toFixed(3) + '</b><br>Parâmetros Treináveis: <b>' + pTr.toFixed(2) + ' M</b>';
    }
    fEl.addEventListener('input', render);
    render();
  }
  function tryInit(){ var r = document.getElementById('sim-ep0905'); if(r) init(r); else setTimeout(tryInit, 200); }
  tryInit();
})();
</script>
''')